# Week 15 V2: Fixed Retrieval Experiment

## Issues Fixed from V1

| Issue | V1 Problem | V2 Fix |
|-------|------------|--------|
| **Verification** | Ran code (fails on network calls) | Syntax-only check |
| **Retrieval content** | Internal httpx source code | Usage examples + docstrings |
| **Context injection** | Mid-generation disruption | Pre-generation context only |
| **Always-retrieve tracking** | Showed 0 retrievals | Fixed counting |

## New Approach

Instead of injecting context mid-generation (which disrupts the model), we:
1. Use CodeGnosis to DECIDE whether to use retrieval BEFORE generation
2. If CodeGnosis predicts high error probability based on the prompt, provide context upfront
3. This is more practical and doesn't disrupt generation flow

In [1]:
!pip install -q torch transformers accelerate bitsandbytes sentence-transformers scipy numpy pandas matplotlib scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 22.0 MB/s eta 0:00:00


In [2]:
import os
import ast
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import random
from typing import List, Dict, Tuple
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [3]:
# Load model
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    output_hidden_states=True,
    output_attentions=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

embedder = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Models loaded!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/646 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Models loaded!


## Better Retrieval Corpus: Usage Examples, Not Source Code

The httpx source code doesn't help the model write correct usage. We need **examples**.

In [4]:
# Create a corpus of httpx USAGE EXAMPLES (not source code)
HTTPX_EXAMPLES = [
    {
        'topic': 'basic GET request',
        'code': '''import httpx

# Simple GET request
response = httpx.get("https://httpbin.org/get")
print(response.status_code)  # 200
print(response.json())  # Response body as dict
'''
    },
    {
        'topic': 'GET with timeout',
        'code': '''import httpx

# GET with timeout (in seconds)
response = httpx.get("https://httpbin.org/get", timeout=10.0)

# Or with Timeout object for fine-grained control
timeout = httpx.Timeout(10.0, connect=5.0)
response = httpx.get("https://httpbin.org/get", timeout=timeout)
'''
    },
    {
        'topic': 'POST with JSON data',
        'code': '''import httpx

# POST JSON data
data = {"name": "test", "value": 123}
response = httpx.post("https://httpbin.org/post", json=data)
print(response.json())
'''
    },
    {
        'topic': 'POST form data',
        'code': '''import httpx

# POST form data
data = {"username": "user", "password": "pass"}
response = httpx.post("https://httpbin.org/post", data=data)
'''
    },
    {
        'topic': 'custom headers',
        'code': '''import httpx

# Custom headers
headers = {"User-Agent": "MyApp/1.0", "Accept": "application/json"}
response = httpx.get("https://httpbin.org/headers", headers=headers)
'''
    },
    {
        'topic': 'Client with base_url',
        'code': '''import httpx

# Using Client for connection pooling
with httpx.Client(base_url="https://httpbin.org") as client:
    response = client.get("/get")
    response2 = client.post("/post", json={"key": "value"})
'''
    },
    {
        'topic': 'handle timeout exception',
        'code': '''import httpx

try:
    response = httpx.get("https://httpbin.org/delay/10", timeout=2.0)
except httpx.TimeoutException:
    print("Request timed out")
except httpx.RequestError as e:
    print(f"Request error: {e}")
'''
    },
    {
        'topic': 'check response status',
        'code': '''import httpx

response = httpx.get("https://httpbin.org/status/404")

# Check if successful (2xx status)
if response.is_success:
    print("Success!")
elif response.is_error:
    print(f"Error: {response.status_code}")

# Or raise exception on error
response.raise_for_status()
'''
    },
    {
        'topic': 'cookies',
        'code': '''import httpx

# Send cookies
cookies = {"session_id": "abc123"}
response = httpx.get("https://httpbin.org/cookies", cookies=cookies)

# Or use Client for automatic cookie handling
with httpx.Client() as client:
    client.cookies.set("session_id", "abc123")
    response = client.get("https://httpbin.org/cookies")
'''
    },
    {
        'topic': 'async client',
        'code': '''import httpx
import asyncio

async def fetch_data():
    async with httpx.AsyncClient() as client:
        response = await client.get("https://httpbin.org/get")
        return response.json()

# Run async function
# result = asyncio.run(fetch_data())
'''
    },
    {
        'topic': 'async multiple requests',
        'code': '''import httpx
import asyncio

async def fetch_multiple(urls):
    async with httpx.AsyncClient() as client:
        tasks = [client.get(url) for url in urls]
        responses = await asyncio.gather(*tasks)
        return [r.json() for r in responses]

# urls = ["https://httpbin.org/get"] * 3
# results = asyncio.run(fetch_multiple(urls))
'''
    },
    {
        'topic': 'handle HTTP errors',
        'code': '''import httpx

response = httpx.get("https://httpbin.org/status/404")

if response.status_code == 404:
    print("Not found")
elif response.status_code >= 400:
    print(f"Error: {response.status_code}")

# Or use raise_for_status with try/except
try:
    response.raise_for_status()
except httpx.HTTPStatusError as e:
    print(f"HTTP error: {e.response.status_code}")
'''
    },
    {
        'topic': 'BasicAuth authentication',
        'code': '''import httpx

# Basic authentication
auth = httpx.BasicAuth("username", "password")
response = httpx.get("https://httpbin.org/basic-auth/username/password", auth=auth)

# Or with Client
with httpx.Client(auth=auth) as client:
    response = client.get("https://httpbin.org/basic-auth/username/password")
'''
    },
    {
        'topic': 'follow redirects',
        'code': '''import httpx

# By default, httpx does NOT follow redirects
# Enable with follow_redirects=True
response = httpx.get(
    "https://httpbin.org/redirect/3",
    follow_redirects=True
)

# Or configure on Client
with httpx.Client(follow_redirects=True) as client:
    response = client.get("https://httpbin.org/redirect/3")
'''
    },
    {
        'topic': 'streaming response',
        'code': '''import httpx

# Stream large response
with httpx.stream("GET", "https://httpbin.org/stream/10") as response:
    for chunk in response.iter_bytes():
        print(chunk)

# Or stream text
with httpx.stream("GET", "https://httpbin.org/stream/10") as response:
    for line in response.iter_lines():
        print(line)
'''
    },
    {
        'topic': 'retry with backoff',
        'code': '''import httpx
import time

def request_with_retry(url, max_retries=3, backoff_factor=2):
    for attempt in range(max_retries):
        try:
            response = httpx.get(url, timeout=10.0)
            response.raise_for_status()
            return response
        except (httpx.RequestError, httpx.HTTPStatusError) as e:
            if attempt == max_retries - 1:
                raise
            wait_time = backoff_factor ** attempt
            print(f"Retry {attempt + 1} after {wait_time}s")
            time.sleep(wait_time)
'''
    },
    {
        'topic': 'file upload multipart',
        'code': '''import httpx

# Upload file
files = {"file": open("document.txt", "rb")}
response = httpx.post("https://httpbin.org/post", files=files)

# With filename and content type
files = {"file": ("report.pdf", open("report.pdf", "rb"), "application/pdf")}
response = httpx.post("https://httpbin.org/post", files=files)
'''
    },
    {
        'topic': 'event hooks',
        'code': '''import httpx
import time

def log_request(request):
    print(f"Request: {request.method} {request.url}")

def log_response(response):
    print(f"Response: {response.status_code}")

# Use event_hooks
with httpx.Client(
    event_hooks={"request": [log_request], "response": [log_response]}
) as client:
    response = client.get("https://httpbin.org/get")
'''
    },
]

print(f"Created {len(HTTPX_EXAMPLES)} usage examples")

# Embed examples
example_texts = [f"{ex['topic']}: {ex['code']}" for ex in HTTPX_EXAMPLES]
EXAMPLE_EMBEDDINGS = embedder.encode(example_texts, convert_to_tensor=True)
print(f"Embeddings shape: {EXAMPLE_EMBEDDINGS.shape}")

Created 18 usage examples
Embeddings shape: torch.Size([18, 384])


In [5]:
def retrieve_examples(query: str, top_k: int = 2) -> str:
    """Retrieve relevant usage examples."""
    query_embedding = embedder.encode(query, convert_to_tensor=True)
    similarities = F.cosine_similarity(query_embedding.unsqueeze(0), EXAMPLE_EMBEDDINGS)
    top_indices = torch.topk(similarities, k=min(top_k, len(HTTPX_EXAMPLES))).indices

    examples = []
    for idx in top_indices:
        ex = HTTPX_EXAMPLES[idx.item()]
        examples.append(f"# Example: {ex['topic']}\n{ex['code']}")

    return "\n\n".join(examples)

# Test
print(retrieve_examples("timeout exception handling"))

# Example: handle timeout exception
import httpx

try:
    response = httpx.get("https://httpbin.org/delay/10", timeout=2.0)
except httpx.TimeoutException:
    print("Request timed out")
except httpx.RequestError as e:
    print(f"Request error: {e}")


# Example: retry with backoff
import httpx
import time

def request_with_retry(url, max_retries=3, backoff_factor=2):
    for attempt in range(max_retries):
        try:
            response = httpx.get(url, timeout=10.0)
            response.raise_for_status()
            return response
        except (httpx.RequestError, httpx.HTTPStatusError) as e:
            if attempt == max_retries - 1:
                raise
            wait_time = backoff_factor ** attempt
            print(f"Retry {attempt + 1} after {wait_time}s")
            time.sleep(wait_time)



## Fixed Verification: Syntax Only

Don't try to run httpx code - it will fail without network access.

In [6]:
def verify_syntax_only(code: str) -> bool:
    """Check if code has valid Python syntax."""
    try:
        ast.parse(code)
        return True
    except SyntaxError:
        return False


def verify_code_quality(code: str, task: str) -> Tuple[bool, str]:
    """Check code quality with multiple criteria."""
    # 1. Syntax check
    if not verify_syntax_only(code):
        return False, "syntax_error"

    # 2. Basic sanity checks for httpx code
    if 'httpx' in task.lower():
        if 'import httpx' not in code and 'from httpx' not in code:
            return False, "missing_import"

        # Check for common mistakes
        if 'requests.' in code:  # Using requests instead of httpx
            return False, "wrong_library"

    # 3. Check for obvious incomplete code
    if code.strip().endswith(':') or code.strip().endswith(','):
        return False, "incomplete"

    return True, "ok"

## Simplified Approach: Predict BEFORE Generation

Instead of interrupting generation, we use task difficulty prediction to decide upfront.

In [7]:
# Train a simple task difficulty predictor based on prompt features
# (This is simpler than CodeGnosis but tests if retrieval helps at all)

def extract_prompt_features(prompt: str) -> Dict[str, float]:
    """Extract features from the prompt to predict difficulty."""
    prompt_lower = prompt.lower()

    return {
        'has_httpx': float('httpx' in prompt_lower),
        'has_async': float('async' in prompt_lower),
        'has_error': float('error' in prompt_lower or 'exception' in prompt_lower),
        'has_timeout': float('timeout' in prompt_lower),
        'has_stream': float('stream' in prompt_lower),
        'has_auth': float('auth' in prompt_lower),
        'has_retry': float('retry' in prompt_lower),
        'has_upload': float('upload' in prompt_lower or 'file' in prompt_lower),
        'has_hook': float('hook' in prompt_lower or 'event' in prompt_lower),
        'word_count': float(len(prompt.split())),
        'is_complex': float(any(w in prompt_lower for w in ['implement', 'retry', 'backoff', 'multipart', 'concurrent'])),
    }

PROMPT_FEATURE_NAMES = list(extract_prompt_features("test").keys())
print(f"Prompt features: {PROMPT_FEATURE_NAMES}")

Prompt features: ['has_httpx', 'has_async', 'has_error', 'has_timeout', 'has_stream', 'has_auth', 'has_retry', 'has_upload', 'has_hook', 'word_count', 'is_complex']


In [8]:
def generate_code(model, tokenizer, prompt: str, context: str = None, max_tokens: int = 200) -> str:
    """Generate code with optional context."""
    if context:
        full_prompt = f"""[INST] Here are some relevant examples:

{context}

Now write code for: {prompt}

Write only Python code, no explanations. [/INST]

```python
"""
    else:
        full_prompt = f"[INST] {prompt}\n\nWrite only Python code, no explanations. [/INST]\n\n```python\n"

    input_ids = tokenizer.encode(full_prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)
    code = generated.replace('```python', '').replace('```', '').strip()

    return code

## Evaluation Tasks

In [9]:
EVAL_TASKS = [
    # httpx tasks (should benefit from examples)
    {'prompt': 'Write code using httpx to make a GET request with 10 second timeout', 'category': 'httpx'},
    {'prompt': 'Write code using httpx to POST JSON data to a URL', 'category': 'httpx'},
    {'prompt': 'Write code using httpx to POST form data', 'category': 'httpx'},
    {'prompt': 'Write code using httpx Client with base_url', 'category': 'httpx'},
    {'prompt': 'Write code using httpx to catch TimeoutException', 'category': 'httpx'},
    {'prompt': 'Write code using httpx to set custom headers', 'category': 'httpx'},
    {'prompt': 'Write code using httpx to check if response is_success', 'category': 'httpx'},
    {'prompt': 'Write code using httpx to send cookies', 'category': 'httpx'},
    {'prompt': 'Write code using httpx AsyncClient', 'category': 'httpx'},
    {'prompt': 'Write code using httpx to handle 404 status', 'category': 'httpx'},
    {'prompt': 'Write code using httpx BasicAuth', 'category': 'httpx'},
    {'prompt': 'Write code using httpx to follow redirects', 'category': 'httpx'},
    {'prompt': 'Write code using httpx to stream response', 'category': 'httpx'},
    {'prompt': 'Write code using httpx with retry and backoff', 'category': 'httpx'},
    {'prompt': 'Write code using httpx to upload a file', 'category': 'httpx'},

    # General tasks (control - shouldn't need httpx examples)
    {'prompt': 'Write a function to find GCD of two numbers', 'category': 'general'},
    {'prompt': 'Write a function to check if list is sorted', 'category': 'general'},
    {'prompt': 'Write a function to reverse a string', 'category': 'general'},
    {'prompt': 'Write a function to find factorial', 'category': 'general'},
    {'prompt': 'Write a function to check if palindrome', 'category': 'general'},
]

print(f"Evaluation tasks: {len(EVAL_TASKS)}")

Evaluation tasks: 20


## Run Experiment

In [10]:
%%time

results = {'none': [], 'always': [], 'smart': []}

for i, task in enumerate(EVAL_TASKS):
    print(f"\n[{i+1}/{len(EVAL_TASKS)}] {task['prompt'][:50]}...")

    # Condition 1: No retrieval
    code_none = generate_code(model, tokenizer, task['prompt'], context=None)
    is_valid_none, reason_none = verify_code_quality(code_none, task['prompt'])
    results['none'].append({'task': task['prompt'], 'category': task['category'],
                            'is_correct': is_valid_none, 'reason': reason_none})

    # Condition 2: Always retrieve (for httpx tasks)
    if 'httpx' in task['prompt'].lower():
        context = retrieve_examples(task['prompt'], top_k=2)
    else:
        context = None
    code_always = generate_code(model, tokenizer, task['prompt'], context=context)
    is_valid_always, reason_always = verify_code_quality(code_always, task['prompt'])
    results['always'].append({'task': task['prompt'], 'category': task['category'],
                              'is_correct': is_valid_always, 'reason': reason_always,
                              'retrieved': context is not None})

    # Condition 3: Smart retrieval (only for complex httpx tasks)
    prompt_features = extract_prompt_features(task['prompt'])
    is_complex = prompt_features['is_complex'] or prompt_features['has_retry'] or \
                 prompt_features['has_stream'] or prompt_features['has_upload'] or \
                 prompt_features['has_hook']

    if 'httpx' in task['prompt'].lower() and is_complex:
        context = retrieve_examples(task['prompt'], top_k=2)
    else:
        context = None
    code_smart = generate_code(model, tokenizer, task['prompt'], context=context)
    is_valid_smart, reason_smart = verify_code_quality(code_smart, task['prompt'])
    results['smart'].append({'task': task['prompt'], 'category': task['category'],
                             'is_correct': is_valid_smart, 'reason': reason_smart,
                             'retrieved': context is not None})

    # Print status
    status_none = "OK" if is_valid_none else f"ERR({reason_none})"
    status_always = "OK" if is_valid_always else f"ERR({reason_always})"
    status_smart = "OK" if is_valid_smart else f"ERR({reason_smart})"
    smart_ret = " (ret)" if results['smart'][-1]['retrieved'] else ""

    print(f"  none:   {status_none}")
    print(f"  always: {status_always}")
    print(f"  smart:  {status_smart}{smart_ret}")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



[1/20] Write code using httpx to make a GET request with ...
  none:   OK
  always: OK
  smart:  OK

[2/20] Write code using httpx to POST JSON data to a URL...
  none:   OK
  always: OK
  smart:  OK

[3/20] Write code using httpx to POST form data...
  none:   OK
  always: OK
  smart:  OK

[4/20] Write code using httpx Client with base_url...
  none:   OK
  always: OK
  smart:  OK

[5/20] Write code using httpx to catch TimeoutException...
  none:   OK
  always: OK
  smart:  OK

[6/20] Write code using httpx to set custom headers...
  none:   OK
  always: OK
  smart:  OK

[7/20] Write code using httpx to check if response is_suc...
  none:   OK
  always: OK
  smart:  OK

[8/20] Write code using httpx to send cookies...
  none:   OK
  always: OK
  smart:  OK

[9/20] Write code using httpx AsyncClient...
  none:   OK
  always: OK
  smart:  OK

[10/20] Write code using httpx to handle 404 status...
  none:   OK
  always: OK
  smart:  OK

[11/20] Write code using httpx BasicAuth...
  non

In [11]:
# Results summary
print("="*70)
print("RESULTS SUMMARY")
print("="*70)

for condition in ['none', 'always', 'smart']:
    correct = sum(1 for r in results[condition] if r['is_correct'])
    total = len(results[condition])
    retrievals = sum(1 for r in results[condition] if r.get('retrieved', False))
    print(f"\n{condition.upper()}:")
    print(f"  Accuracy: {correct}/{total} ({100*correct/total:.1f}%)")
    print(f"  Retrievals: {retrievals}")

# By category
print("\n" + "="*70)
print("BY CATEGORY")
print("="*70)

for cat in ['httpx', 'general']:
    print(f"\n{cat.upper()}:")
    for condition in ['none', 'always', 'smart']:
        cat_results = [r for r in results[condition] if r['category'] == cat]
        correct = sum(1 for r in cat_results if r['is_correct'])
        total = len(cat_results)
        if total > 0:
            print(f"  {condition:8}: {correct}/{total} ({100*correct/total:.0f}%)")

RESULTS SUMMARY

NONE:
  Accuracy: 19/20 (95.0%)
  Retrievals: 0

ALWAYS:
  Accuracy: 18/20 (90.0%)
  Retrievals: 15

SMART:
  Accuracy: 18/20 (90.0%)
  Retrievals: 3

BY CATEGORY

HTTPX:
  none    : 14/15 (93%)
  always  : 13/15 (87%)
  smart   : 13/15 (87%)

GENERAL:
  none    : 5/5 (100%)
  always  : 5/5 (100%)
  smart   : 5/5 (100%)


In [12]:
# Compare none vs always for httpx tasks
print("\n" + "="*70)
print("DOES RETRIEVAL HELP FOR HTTPX TASKS?")
print("="*70)

httpx_none = [r for r in results['none'] if r['category'] == 'httpx']
httpx_always = [r for r in results['always'] if r['category'] == 'httpx']

none_correct = sum(1 for r in httpx_none if r['is_correct'])
always_correct = sum(1 for r in httpx_always if r['is_correct'])

print(f"\nWithout retrieval: {none_correct}/{len(httpx_none)} ({100*none_correct/len(httpx_none):.0f}%)")
print(f"With retrieval:    {always_correct}/{len(httpx_always)} ({100*always_correct/len(httpx_always):.0f}%)")
print(f"Improvement:       {always_correct - none_correct:+d} tasks")

if always_correct > none_correct:
    print("\n=> RETRIEVAL HELPS! Examples improve httpx code generation.")
elif always_correct == none_correct:
    print("\n=> NO DIFFERENCE. Retrieval neither helps nor hurts.")
else:
    print("\n=> RETRIEVAL HURTS. Examples confuse the model.")


DOES RETRIEVAL HELP FOR HTTPX TASKS?

Without retrieval: 14/15 (93%)
With retrieval:    13/15 (87%)
Improvement:       -1 tasks

=> RETRIEVAL HURTS. Examples confuse the model.


In [13]:
# Detailed per-task comparison
print("\n" + "="*70)
print("PER-TASK COMPARISON (httpx only)")
print("="*70)

for i, (none_r, always_r) in enumerate(zip(httpx_none, httpx_always)):
    none_status = "OK" if none_r['is_correct'] else "ERR"
    always_status = "OK" if always_r['is_correct'] else "ERR"

    if none_r['is_correct'] != always_r['is_correct']:
        change = "FIXED" if always_r['is_correct'] else "BROKE"
        print(f"\n{none_r['task'][:60]}...")
        print(f"  None: {none_status}, Always: {always_status} -> {change}")


PER-TASK COMPARISON (httpx only)

Write code using httpx with retry and backoff...
  None: OK, Always: ERR -> BROKE


In [14]:
# Save results
output = {
    'experiment': 'Week15_V2_Fixed_Retrieval',
    'conditions': {
        cond: {
            'accuracy': sum(1 for r in res if r['is_correct']) / len(res),
            'total': len(res),
            'retrievals': sum(1 for r in res if r.get('retrieved', False)),
        }
        for cond, res in results.items()
    },
}

with open('week15_v2_results.json', 'w') as f:
    json.dump(output, f, indent=2)

print("Results saved!")

Results saved!
